# Donkey Kong Inverso — Monte Carlo vs Q-Learning
### Práctica de Reinforcement Learning Clásico

| | |
|---|---|
| **Autor MC** | Miguel J. Gutiérrez — Monte Carlo on-policy |
| **Autor QL** | Adrián Pavón — Q-Learning off-policy |
| **Entorno** | DonkeyKongInverso 6×6, determinista y estocástico |
| **Algoritmos** | Monte Carlo · Q-Learning |

## Instalación de dependencias

In [ ]:
# ── Instalación de dependencias ───────────────────────────────────────────────
# Todas las librerías usadas forman parte de la distribución estándar de
# Python científico. En la mayoría de entornos (Colab, Anaconda, pip base)
# ya están disponibles. Si no, ejecutar esta celda las instala.

import subprocess, sys

_required = ["numpy", "matplotlib"]

for pkg in _required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print(" Dependencias verificadas.")

# Imports y sistema de diseño visual

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch
from collections import defaultdict
import random
import warnings

warnings.filterwarnings("ignore")

# ── Semilla global ────────────────────────────────────────────────────────────
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# ══════════════════════════════════════════════════════════════════════════════
#  DESIGN SYSTEM — paleta retro-arcade, dark theme
#  Modificar aquí para cambiar la estética de TODAS las figuras.
# ══════════════════════════════════════════════════════════════════════════════
DS = {
    # Fondo
    "bg_fig"      : "#0D0D0D",   # fondo de figura (casi negro)
    "bg_ax"       : "#111827",   # fondo de ejes (azul muy oscuro)
    "bg_cell"     : "#1E293B",   # celda libre en el mapa

    # Colores de celda
    "col_start"   : "#1D4ED8",   # azul (inicio)
    "col_goal"    : "#16A34A",   # verde (meta)
    "col_hole"    : "#DC2626",   # rojo (agujero)
    "col_ladder"  : "#D97706",   # ámbar (escalera)
    "col_agent"   : "#F0FFF4",   # blanco verdoso (agente)
    "col_path"    : "#A78BFA",   # violeta (ruta óptima)

    # Curvas de aprendizaje
    "col_mc"      : "#38BDF8",   # azul neón (Monte Carlo)
    "col_ql"      : "#F472B6",   # rosa neón (Q-Learning)
    "col_random"  : "#6B7280",   # gris (baseline aleatorio)

    # Texto y rejilla
    "col_text"    : "#F1F5F9",   # blanco suave
    "col_subtext" : "#94A3B8",   # gris claro
    "col_grid"    : "#1E293B",   # rejilla muy sutil
    "col_arrow"   : "#E2E8F0",   # flechas de política

    # Tipografía
    "font_mono"   : "DejaVu Sans Mono",   # disponible en Colab sin instalación
    "font_sans"   : "DejaVu Sans",

    # Grosor de líneas
    "lw_curve"    : 2.0,
    "lw_smooth"   : 2.5,
    "lw_grid"     : 0.4,
}

# ── Aplicar estilo global a matplotlib ───────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor"    : DS["bg_fig"],
    "axes.facecolor"      : DS["bg_ax"],
    "axes.edgecolor"      : DS["col_subtext"],
    "axes.labelcolor"     : DS["col_text"],
    "axes.titlecolor"     : DS["col_text"],
    "xtick.color"         : DS["col_subtext"],
    "ytick.color"         : DS["col_subtext"],
    "text.color"          : DS["col_text"],
    "grid.color"          : DS["col_grid"],
    "grid.linewidth"      : DS["lw_grid"],
    "legend.facecolor"    : "#1E293B",
    "legend.edgecolor"    : DS["col_subtext"],
    "legend.labelcolor"   : DS["col_text"],
    "font.family"         : DS["font_sans"],
    "font.size"           : 11,
    "axes.titlesize"      : 13,
    "axes.labelsize"      : 11,
    "figure.dpi"          : 130,
    "savefig.dpi"         : 150,
    "savefig.facecolor"   : DS["bg_fig"],
})

print(" Design system cargado.")
print(f" Paleta: retro-arcade dark | Fuente mono: {DS['font_mono']}")

# Configuración centralizada (CONFIG)

In [ ]:
# ── Configuración global del experimento ──────────────────────────────────────
CONFIG = {
    "rows": 6, "cols": 6,
    "n_episodes"       : 5000,
    "n_runs"           : 5,
    "seeds"            : [42, 7, 13, 99, 2025],
    "gamma"            : 0.99,
    "eps_start"        : 0.20,
    "eps_end"          : 0.01,
    "eps_decay"        : 0.999,
    "alpha"            : 0.10,   # solo Q-Learning
    "smoothing_window" : 100,
    "eval_last_n"      : 500,
}

def decay_epsilon(eps: float) -> float:
    """Aplica decaimiento multiplicativo respetando el mínimo configurado."""
    return max(CONFIG["eps_end"], eps * CONFIG["eps_decay"])

print(" CONFIG:", CONFIG)

# Entorno DonkeyKongInverso (versión mejorada)

In [ ]:
class DonkeyKongInverso:
    """
    Entorno tipo plataformas 6x6 inspirado en Donkey Kong inverso.

    Mapa:
        S . . . . .
        . . L . L .
        . H . H . .
        . . . . . .
        . . L . L .
        . . . . . G

    Leyenda:
        S = inicio (0,0)   G = meta (5,5)
        H = agujero        L = escalera
        . = celda libre

    Acciones: 0=arriba, 1=abajo, 2=izquierda, 3=derecha

    Movimiento vertical: solo funciona si la celda actual es una escalera.
    Si no lo es, la acción no produce desplazamiento (y cuesta -1 igualmente).

    Recompensas:
        Meta alcanzada : +20, done=True
        Agujero        :  -1, done=True
        Cualquier paso :  -1, done=False

    Parámetro stochastic:
        Si True, con probabilidad 0.10 la acción elegida se ignora
        y el agente permanece en la misma celda (sigue costando -1).
        Permite estudiar robustez de MC vs QL ante ruido.
    """

    # ── Constantes del mapa ───────────────────────────────────────────────────
    _LADDER_PAIRS = {
    (0, 2): (3, 0), (3, 0): (0, 2),
    (1, 1): (4, 2), (4, 2): (1, 1),
    (1, 4): (5, 1), (5, 1): (1, 4),
    (2, 5): (4, 5), (4, 5): (2, 5),
    (3, 3): (5, 3), (5, 3): (3, 3),
    }

    _HOLES   = [(2, 1), (2, 3)]
    _START   = (0, 0)
    _GOAL    = (5, 5)
    _ROWS, _COLS = 6, 6

    # Símbolos para render()
    _SYMBOLS = {
        "start"  : "🟦",
        "goal"   : "🟩",
        "hole"   : "🟥",
        "ladder" : "🪜",
        "empty"  : "⬜",
        "agent"  : "🤖",
    }
    _ARROW = ["↑", "↓", "←", "→"]

    def __init__(self, stochastic: bool = False, slip_prob: float = 0.10):
        """
        Parameters
        ----------
        stochastic : bool
            Si True, activa ruido en el entorno (slip_prob de quedarse quieto).
        slip_prob : float
            Probabilidad de que la acción sea ignorada (solo si stochastic=True).
        """
        self.rows         = self._ROWS
        self.cols         = self._COLS
        self.start        = self._START
        self.goal         = self._GOAL
        self.ladders      = self._LADDER_PAIRS
        self.holes        = self._HOLES
        self.action_space = 4
        self.stochastic   = stochastic
        self.slip_prob    = slip_prob
        self.state        = None
        self.n_states     = self.rows * self.cols   # 36 estados totales

    # ── Conversión estado ↔ índice ────────────────────────────────────────────
    def state_to_idx(self, state: tuple) -> int:
        """Convierte (fila, columna) a un índice lineal único [0, 35]."""
        r, c = state
        return r * self.cols + c

    def idx_to_state(self, idx: int) -> tuple:
        """Convierte índice lineal a (fila, columna)."""
        return (idx // self.cols, idx % self.cols)

    # ── Control de episodio ───────────────────────────────────────────────────
    def reset(self) -> tuple:
        """Reinicia el entorno al estado inicial. Devuelve el estado (fila, col)."""
        self.state = self.start
        return self.state

    def step(self, action: int):
        """
        Ejecuta una acción y devuelve (next_state, reward, done).

        Parameters
        ----------
        action : int  ->  0=arriba, 1=abajo, 2=izquierda, 3=derecha
        """
        # Ruido estocástico: con slip_prob la acción se ignora
        if self.stochastic and np.random.random() < self.slip_prob:
            new_state = self.state          # el agente se queda quieto
            reward    = -1                  # igualmente cuesta un paso
            done      = False
            self.state = new_state
            return new_state, reward, done

        r, c = self.state

        # Movimiento horizontal: siempre permitido (con bordes del grid)
        if action == 2:    # izquierda
            c = max(c - 1, 0)
        elif action == 3:  # derecha
            c = min(c + 1, self.cols - 1)

        # Movimiento vertical: solo si la celda actual es una escalera
        elif action == 0:  # arriba
            if (r, c) in self.ladders:
                r, c = self.ladders[(r, c)]
            # si no hay escalera, la posición no cambia
        elif action == 1:  # abajo
            if (r, c) in self.ladders:
                r, c = self.ladders[(r, c)]
            # si no hay escalera, la posición no cambia

        new_state = (r, c)

        # Evaluación de la nueva celda
        if new_state == self.goal:
            reward, done = 20, True
        elif new_state in self.holes:
            reward, done = -1, True
        else:
            reward, done = -1, False

        self.state = new_state
        return new_state, reward, done

    # ── Visualización del mapa ────────────────────────────────────────────────
    def render(self, agent_pos: tuple = None, policy_Q: np.ndarray = None) -> None:
        """
        Imprime el mapa en consola con emojis.

        Parameters
        ----------
        agent_pos : tuple, opcional
            Si se proporciona, dibuja el agente (🤖) en esa posición.
        policy_Q  : np.ndarray, opcional
            Tabla Q de shape (n_states, 4). Si se proporciona, muestra flechas
            de la política greedy en cada celda libre.
        """
        print()
        header = "    " + "  ".join([f" {c} " for c in range(self.cols)])
        print(header)
        print("   " + "─" * (self.cols * 4))

        for r in range(self.rows):
            row_str = f" {r} │"
            for c in range(self.cols):
                pos = (r, c)
                if pos == agent_pos:
                    cell = self._SYMBOLS["agent"]
                elif pos == self.goal:
                    cell = self._SYMBOLS["goal"]
                elif pos in self.holes:
                    cell = self._SYMBOLS["hole"]
                elif pos in self.ladders:
                    cell = self._SYMBOLS["ladder"]
                elif pos == self.start:
                    cell = self._SYMBOLS["start"]
                elif policy_Q is not None:
                    # muestra la flecha de la acción greedy
                    idx    = self.state_to_idx(pos)
                    best_a = int(np.argmax(policy_Q[idx]))
                    cell   = f" {self._ARROW[best_a]} "
                else:
                    cell = self._SYMBOLS["empty"]
                row_str += cell
            print(row_str)
        print()

    def get_map_matrix(self) -> list:
        """Devuelve el mapa como lista de listas de caracteres (para plots)."""
        grid = [["." for _ in range(self.cols)] for _ in range(self.rows)]
        grid[self.start[0]][self.start[1]] = "S"
        grid[self.goal[0]][self.goal[1]]   = "G"
        for h in self.holes:
            grid[h[0]][h[1]] = "H"
        for ladder in self.ladders:
            grid[ladder[0]][ladder[1]] = "L"
        return grid


# ── Instancias del entorno ────────────────────────────────────────────────────
env            = DonkeyKongInverso(stochastic=False)  # determinista (principal)
env_stochastic = DonkeyKongInverso(stochastic=True)   # estocástico (pregunta 4)

print(" Entorno determinista  creado. Estados:", env.n_states)
print(" Entorno estocástico   creado. Estados:", env_stochastic.n_states)
print()
print("── Mapa inicial ─────────────────────────────────────────────")
env.render()

# Funciones helper compartidas

In [ ]:
# ── Helpers compartidos ───────────────────────────────────────────────────────

def eps_greedy_action(Q: np.ndarray, state_idx: int, epsilon: float) -> int:
    """
    Selecciona una acción con política ε-greedy.

    Con probabilidad ε elige una acción aleatoria (exploración).
    Con probabilidad 1-ε elige la acción con mayor Q[s] (explotación).

    Parameters
    ----------
    Q         : np.ndarray, shape (n_states, 4)
    state_idx : int, índice lineal del estado actual
    epsilon   : float, probabilidad de exploración en [0, 1]
    """
    if np.random.random() < epsilon:
        return np.random.randint(4)             # exploración aleatoria uniforme
    return int(np.argmax(Q[state_idx]))         # explotación greedy


def smooth(values: list, window: int = 100) -> np.ndarray:
    """
    Aplica una media móvil de tamaño `window` sobre una lista de valores.
    Se usa para suavizar las curvas de recompensa y facilitar la comparación visual.
    """
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")


def compute_metrics(rewards: list, steps: list, last_n: int = 500) -> dict:
    """
    Calcula métricas de evaluación sobre los últimos `last_n` episodios.

    Returns
    -------
    dict con:
        mean_reward  : recompensa media
        success_rate : fracción de episodios que llegaron a la meta
        mean_steps   : número medio de pasos por episodio exitoso
    """
    r_tail = rewards[-last_n:]
    s_tail = steps[-last_n:]

    # Un episodio se considera exitoso si la recompensa > umbral de agujero.
    # La recompensa mínima alcanzando la meta con 1 paso sería 20-1=19;
    # cayendo en agujero en el primer paso sería -1. Umbral conservador: 0.
    successes    = [r for r in r_tail if r > 0]
    success_rate = len(successes) / len(r_tail)

    # Pasos medios solo sobre episodios exitosos (llegar a meta)
    success_steps = [s for r, s in zip(r_tail, s_tail) if r > 0]
    mean_steps    = np.mean(success_steps) if success_steps else float("nan")

    return {
        "mean_reward"  : np.mean(r_tail),
        "success_rate" : success_rate,
        "mean_steps"   : mean_steps,
    }


def init_Q(n_states: int, n_actions: int = 4) -> np.ndarray:
    """
    Inicializa la tabla Q a ceros.
    Shape: (n_states, n_actions) = (36, 4).

    Se inicializa a cero (valor neutro) en lugar de valores optimistas
    para no sesgar la exploración en este entorno pequeño.
    """
    return np.zeros((n_states, n_actions))


print(" Funciones helper definidas:")
print("   eps_greedy_action | smooth | compute_metrics | init_Q")

# Función de visualización

In [ ]:
def plot_map(env,
             Q          : np.ndarray = None,
             path       : list       = None,
             title      : str        = "Mapa del entorno",
             show_policy: bool       = True) -> None:
    """
    Renderiza el mapa del entorno con estética retro-arcade.

    Parameters
    ----------
    env         : DonkeyKongInverso
    Q           : np.ndarray (n_states, 4), opcional.
                  Si se proporciona y show_policy=True, dibuja flechas greedy.
    path        : list de tuplas (r,c), opcional.
                  Si se proporciona, resalta la ruta del agente.
    title       : str, título de la figura.
    show_policy : bool, si mostrar flechas de política.
    """
    rows, cols = env.rows, env.cols
    ARROW_MAP  = {0: (0, -0.35), 1: (0, 0.35), 2: (-0.35, 0), 3: (0.35, 0)}
    # (dx, dy) en coordenadas de celda para cada acción (0↑ 1↓ 2← 3→)
    # Nota: eje y de matplotlib va hacia abajo, por eso ↑ tiene dy negativo.

    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor(DS["bg_fig"])
    ax.set_facecolor(DS["bg_fig"])

    # ── Dibujar celdas ────────────────────────────────────────────────────────
    for r in range(rows):
        for c in range(cols):
            pos = (r, c)

            # Determinar color de fondo de celda
            if pos == env.goal:
                fc = DS["col_goal"]
            elif pos in env.holes:
                fc = DS["col_hole"]
            elif pos in env.ladders:
                fc = DS["col_ladder"]
            elif pos == env.start:
                fc = DS["col_start"]
            else:
                fc = DS["bg_cell"]

            # Rectángulo de celda con borde redondeado
            rect = FancyBboxPatch(
                (c - 0.46, r - 0.46), 0.92, 0.92,
                boxstyle="round,pad=0.04",
                facecolor=fc,
                edgecolor=DS["bg_fig"],
                linewidth=2.0,
                zorder=1,
            )
            ax.add_patch(rect)

            # ── Etiqueta de celda ─────────────────────────────────────────────
            if pos == env.start:
                label, fs, fw = "S", 15, "bold"
            elif pos == env.goal:
                label, fs, fw = "G", 15, "bold"
            elif pos in env.holes:
                label, fs, fw = "✕", 16, "bold"
            elif pos in env.ladders:
                label, fs, fw = "╠╣", 11, "normal"
            else:
                label, fs, fw = "", 10, "normal"

            if label:
                ax.text(c, r, label,
                        ha="center", va="center",
                        fontsize=fs, fontweight=fw,
                        fontfamily=DS["font_mono"],
                        color=DS["col_text"],
                        zorder=3,
                        path_effects=[pe.withStroke(linewidth=2,
                                                    foreground=DS["bg_fig"])])

    # ── Ruta óptima (si se proporciona) ──────────────────────────────────────
    if path:
        for i in range(len(path) - 1):
            r0, c0 = path[i]
            r1, c1 = path[i + 1]
            ax.annotate(
                "", xy=(c1, r1), xytext=(c0, r0),
                arrowprops=dict(
                    arrowstyle="->,head_width=0.25,head_length=0.18",
                    color=DS["col_path"],
                    lw=2.5,
                    connectionstyle="arc3,rad=0.0",
                ),
                zorder=5,
            )
        # Resaltar celdas de la ruta con halo
        for r, c in path[1:-1]:
            halo = plt.Circle((c, r), 0.38,
                               color=DS["col_path"], alpha=0.18, zorder=0)
            ax.add_patch(halo)

    # ── Política greedy (flechas) ─────────────────────────────────────────────
    if Q is not None and show_policy:
        for r in range(rows):
            for c in range(cols):
                pos = (r, c)
                if pos in env.holes or pos == env.goal:
                    continue
                s_idx  = env.state_to_idx(pos)
                best_a = int(np.argmax(Q[s_idx]))
                dx, dy = ARROW_MAP[best_a]
                ax.annotate(
                    "", xy=(c + dx, r + dy), xytext=(c - dx, r - dy),
                    arrowprops=dict(
                        arrowstyle="->,head_width=0.2,head_length=0.15",
                        color=DS["col_arrow"],
                        lw=1.4,
                        alpha=0.85,
                    ),
                    zorder=4,
                )

    # ── Leyenda ───────────────────────────────────────────────────────────────
    legend_items = [
        mpatches.Patch(facecolor=DS["col_start"],  label="Inicio (S)"),
        mpatches.Patch(facecolor=DS["col_goal"],   label="Meta (G)"),
        mpatches.Patch(facecolor=DS["col_hole"],   label="Agujero (✕)"),
        mpatches.Patch(facecolor=DS["col_ladder"], label="Escalera (╠╣)"),
        mpatches.Patch(facecolor=DS["bg_cell"],    label="Celda libre"),
    ]
    if path:
        legend_items.append(
            mpatches.Patch(facecolor=DS["col_path"], alpha=0.7, label="Ruta óptima")
        )
    ax.legend(handles=legend_items,
              loc="upper right", fontsize=8.5,
              framealpha=0.85, borderpad=0.8)

    # ── Rejilla de coordenadas ────────────────────────────────────────────────
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_xticklabels([str(c) for c in range(cols)],
                       fontfamily=DS["font_mono"], fontsize=9,
                       color=DS["col_subtext"])
    ax.set_yticklabels([str(r) for r in range(rows)],
                       fontfamily=DS["font_mono"], fontsize=9,
                       color=DS["col_subtext"])
    ax.set_xlim(-0.55, cols - 0.45)
    ax.set_ylim(rows - 0.45, -0.55)   # eje y invertido (fila 0 arriba)
    ax.tick_params(length=0)
    ax.set_xlabel("columna", fontfamily=DS["font_mono"],
                  color=DS["col_subtext"], fontsize=9)
    ax.set_ylabel("fila",    fontfamily=DS["font_mono"],
                  color=DS["col_subtext"], fontsize=9)

    # ── Título con estilo arcade ──────────────────────────────────────────────
    ax.set_title(title, fontfamily=DS["font_mono"],
                 fontsize=14, fontweight="bold",
                 color=DS["col_text"], pad=14,
                 path_effects=[pe.withStroke(linewidth=3,
                                             foreground=DS["bg_fig"])])

    # Borde exterior de la figura
    for spine in ax.spines.values():
        spine.set_edgecolor(DS["col_subtext"])
        spine.set_linewidth(0.8)

    plt.tight_layout()
    plt.show()


# ── Vista previa del mapa vacío ───────────────────────────────────────────────
env = DonkeyKongInverso(stochastic=False)
env_stochastic = DonkeyKongInverso(stochastic=True)

plot_map(env, title="Donkey Kong Inverso — Mapa del entorno")

# Función de curvas de aprendizaje

In [ ]:
def plot_learning_curves(rewards_mc   : np.ndarray = None,
                         rewards_ql   : np.ndarray = None,
                         rewards_rand : np.ndarray = None,
                         std_mc       : np.ndarray = None,
                         std_ql       : np.ndarray = None,
                         title        : str = "Curvas de aprendizaje") -> None:
    """
    Dibuja las curvas de recompensa suavizadas para MC y/o QL.

    Parameters
    ----------
    rewards_mc/ql/rand : np.ndarray 1D, recompensa media por episodio.
    std_mc/ql          : np.ndarray 1D, desviación estándar entre runs.
    title              : str
    """
    w   = CONFIG["smoothing_window"]
    fig, ax = plt.subplots(figsize=(11, 5))

    def _plot_curve(rewards, std, color, label):
        if rewards is None:
            return
        smoothed = smooth(rewards, w)
        x = np.arange(w - 1, len(rewards))

        # Banda de desviación estándar (incertidumbre entre seeds)
        if std is not None:
            std_s = smooth(std, w)
            ax.fill_between(x,
                            smoothed - std_s,
                            smoothed + std_s,
                            alpha=0.15, color=color, linewidth=0)

        # Curva raw (muy tenue, contexto)
        ax.plot(rewards, color=color, alpha=0.12,
                linewidth=DS["lw_curve"] * 0.6, zorder=1)

        # Curva suavizada (protagonista)
        ax.plot(x, smoothed, color=color,
                linewidth=DS["lw_smooth"], label=label, zorder=3)

        # Anotación del valor final
        ax.annotate(f"{smoothed[-1]:.1f}",
                    xy=(x[-1], smoothed[-1]),
                    xytext=(8, 0), textcoords="offset points",
                    color=color, fontsize=9,
                    fontfamily=DS["font_mono"],
                    va="center",
                    path_effects=[pe.withStroke(linewidth=2,
                                                foreground=DS["bg_fig"])])

    _plot_curve(rewards_rand, None,    DS["col_random"], "Aleatorio (baseline)")
    _plot_curve(rewards_mc,   std_mc,  DS["col_mc"],     "Monte Carlo (on-policy)")
    _plot_curve(rewards_ql,   std_ql,  DS["col_ql"],     "Q-Learning (off-policy)")

    # Línea de referencia: recompensa = 0
    ax.axhline(0, color=DS["col_subtext"], linewidth=0.8,
               linestyle="--", alpha=0.5, zorder=0)
    ax.text(10, 1.5, "r = 0", color=DS["col_subtext"],
            fontsize=8, fontfamily=DS["font_mono"], alpha=0.7)

    ax.set_xlabel("Episodio", fontfamily=DS["font_mono"])
    ax.set_ylabel("Recompensa total", fontfamily=DS["font_mono"])
    ax.set_title(title, fontfamily=DS["font_mono"],
                 fontsize=14, fontweight="bold", pad=12)
    ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
    ax.grid(True, axis="y", alpha=0.3)
    ax.grid(True, axis="x", alpha=0.15)

    # Nota sobre suavizado
    ax.text(0.01, 0.02,
            f"Curva suavizada: media móvil {w} eps | Banda: ±1 std ({CONFIG['n_runs']} seeds)",
            transform=ax.transAxes,
            fontsize=7.5, color=DS["col_subtext"],
            fontfamily=DS["font_mono"])

    plt.tight_layout()
    plt.show()


# ── Test visual de la función (curva aleatoria de ejemplo) ────────────────────
_demo = np.random.normal(-15, 5, 5000)   # simula baseline aleatorio
plot_learning_curves(rewards_rand=_demo,
                     title="Demo — estructura de curvas (baseline aleatorio)")
del _demo

# Función de tabla de métricas final

In [ ]:
def print_metrics_table(metrics: dict) -> None:
    """
    Imprime una tabla comparativa de métricas con estilo.

    Parameters
    ----------
    metrics : dict con claves 'MC' y/o 'QL', cada una con el dict
              devuelto por compute_metrics().
    """
    sep   = "─" * 52
    eval_n = CONFIG["eval_last_n"]
    print(f"\n  {'MÉTRICAS FINALES':^48}")
    print(f"  {f'(últimos {eval_n} episodios)':^48}")
    print(f"  {sep}")
    print(f"  {'Métrica':<22} {'Monte Carlo':>12} {'Q-Learning':>12}")
    print(f"  {sep}")

    keys = [
        ("mean_reward",  "Recompensa media",  "{:.2f}"),
        ("success_rate", "Tasa de éxito",     "{:.1%}"),
        ("mean_steps",   "Pasos medios (✓)",  "{:.1f}"),
    ]

    for key, label, fmt in keys:
        mc_val = metrics.get("MC",  {}).get(key, float("nan"))
        ql_val = metrics.get("QL",  {}).get(key, float("nan"))

        mc_str = fmt.format(mc_val) if not np.isnan(mc_val) else "—"
        ql_str = fmt.format(ql_val) if not np.isnan(ql_val) else "—"

        print(f"  {label:<22} {mc_str:>12} {ql_str:>12}")

    print(f"  {sep}")
    print()

    # Umbrales de calidad (entorno determinista)
    print("  UMBRALES DE CALIDAD ESPERADOS (entorno determinista):")
    print("   Tasa de éxito     > 70%  en últimos 500 eps")
    print("   Recompensa media  > 0    en últimos 500 eps")
    print("   Pasos medios      < 20   en últimos 500 eps")
    print("     (ruta óptima ≈ 12 pasos)\n")


# ── Test de la tabla ──────────────────────────────────────────────────────────
_demo_metrics = {
    "MC": {"mean_reward": 5.2, "success_rate": 0.78, "mean_steps": 14.3},
}
print_metrics_table(_demo_metrics)
del _demo_metrics

# Política aleatoria

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Exploración del entorno con política aleatoria
# ════════════════════════════════════════════════════════════════════════════

np.random.seed(GLOBAL_SEED)

N_RANDOM_EPISODES = 20
random_rewards    = []
random_steps_list = []
random_outcomes   = []   # "meta", "agujero", "timeout"
MAX_STEPS         = 200  # límite por episodio para evitar bucles infinitos

print("═" * 55)
print("  Política aleatoria (baseline)")
print("═" * 55)

for ep in range(N_RANDOM_EPISODES):
    state      = env.reset()
    done       = False
    total_r    = 0
    n_steps    = 0
    outcome    = "timeout"

    while not done and n_steps < MAX_STEPS:
        action               = np.random.randint(4)      # acción uniformemente aleatoria
        next_state, r, done  = env.step(action)
        total_r             += r
        n_steps             += 1
        state                = next_state

        if done:
            # Determinar qué terminó el episodio
            if next_state == env.goal:
                outcome = "meta"
            elif next_state in env.holes:
                outcome = "agujero"

    random_rewards.append(total_r)
    random_steps_list.append(n_steps)
    random_outcomes.append(outcome)

    print(f"  Ep {ep+1:>2d} │ pasos={n_steps:>4d} │ "
          f"recompensa={total_r:>6.1f} │ {outcome}")

# ── Resumen numérico ──────────────────────────────────────────────────────────
print()
print(f"  Recompensa media  : {np.mean(random_rewards):.2f}")
print(f"  Recompensa mínima : {np.min(random_rewards):.2f}")
print(f"  Recompensa máxima : {np.max(random_rewards):.2f}")
print(f"  Llegó a meta      : {random_outcomes.count('meta')}/{N_RANDOM_EPISODES}")
print(f"  Cayó en agujero   : {random_outcomes.count('agujero')}/{N_RANDOM_EPISODES}")
print(f"  Timeout (>{MAX_STEPS} pasos) : {random_outcomes.count('timeout')}/{N_RANDOM_EPISODES}")
print()

# Visualización

In [ ]:
# ── Gráfica de resultados de la política aleatoria ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel izquierdo: recompensa por episodio
ax = axes[0]
colors_bar = [DS["col_goal"]   if o == "meta"    else
              DS["col_hole"]   if o == "agujero" else
              DS["col_random"] for o in random_outcomes]

bars = ax.bar(range(1, N_RANDOM_EPISODES + 1),
              random_rewards, color=colors_bar,
              edgecolor=DS["bg_fig"], linewidth=0.8, alpha=0.9)

# Línea de media
ax.axhline(np.mean(random_rewards),
           color=DS["col_text"], linewidth=1.2,
           linestyle="--", alpha=0.7, label=f"Media: {np.mean(random_rewards):.1f}")
ax.set_xlabel("Episodio",         fontfamily=DS["font_mono"])
ax.set_ylabel("Recompensa total", fontfamily=DS["font_mono"])
ax.set_title("Política aleatoria — Recompensa por episodio",
             fontfamily=DS["font_mono"], fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, axis="y", alpha=0.3)

# Leyenda de colores
legend_patches = [
    mpatches.Patch(color=DS["col_goal"],   label="Llegó a meta"),
    mpatches.Patch(color=DS["col_hole"],   label="Agujero"),
    mpatches.Patch(color=DS["col_random"], label="Timeout"),
]
ax.legend(handles=legend_patches, fontsize=8, loc="lower left")

# Panel derecho: distribución de pasos
ax2 = axes[1]
ax2.hist(random_steps_list, bins=10,
         color=DS["col_mc"], alpha=0.8,
         edgecolor=DS["bg_fig"], linewidth=0.8)
ax2.axvline(np.mean(random_steps_list),
            color=DS["col_text"], linewidth=1.4,
            linestyle="--", label=f"Media: {np.mean(random_steps_list):.0f} pasos")
ax2.set_xlabel("Pasos por episodio", fontfamily=DS["font_mono"])
ax2.set_ylabel("Frecuencia",         fontfamily=DS["font_mono"])
ax2.set_title("Política aleatoria — Distribución de pasos",
              fontfamily=DS["font_mono"], fontsize=11, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, axis="y", alpha=0.3)

fig.suptitle("Exploración con política aleatoria (baseline)",
             fontfamily=DS["font_mono"], fontsize=13,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Mapa del entorno ──────────────────────────────────────────────────────────
plot_map(env, title="Mapa del entorno — Donkey Kong Inverso")

# Algoritmo Q-LEARNING


In [ ]:
def run_q_learning(env, n_episodes, gamma, eps_start, eps_end, eps_decay, seed=42):
    np.random.seed(seed)

    # Inicializar tabla Q
    Q = init_Q(env.n_states)

    # Parámetros
    alpha = CONFIG["alpha"]
    epsilon = eps_start

    # Listas para métricas
    rewards_history = [] # recompesas
    steps_history = [] # pasos
    epsilons_history = [] # epsilon

    # Límite de seguridad
    MAX_STEPS = 500

    for ep in range(n_episodes):
        state = env.reset()                          # ← tupla (0, 0)
        done = False
        total_reward = 0
        n_steps = 0

        while not done and n_steps < MAX_STEPS:
            # Convertir tupla a índice
            s_idx = env.state_to_idx(state)          # ← NECESARIO

            # Seleccionar acción ε-greedy
            action = eps_greedy_action(Q, s_idx, epsilon)  # ← usa la función helper

            # Ejecutar paso
            next_state, reward, done = env.step(action)   # ← 3 valores, no 5

            # Acumular recompensa
            total_reward += reward
            n_steps += 1

            # Convertir siguiente estado a índice
            s_next_idx = env.state_to_idx(next_state)     # ← NECESARIO

            # La actualización de Q-Learning (aquí es la magia)
            mejor_valor_siguiente = np.max(Q[s_next_idx]) if not done else 0
            objetivo = reward + gamma * mejor_valor_siguiente
            Q[s_idx, action] += alpha * (objetivo - Q[s_idx, action])

            # Siguiente iteración
            state = next_state

        # Fin del episodio
        rewards_history.append(total_reward)
        steps_history.append(n_steps)
        epsilons_history.append(epsilon)
        epsilon = decay_epsilon(epsilon)

         #RETORNAR RESULTADOS
    return {
        "Q"       : Q,
        "rewards" : rewards_history,
        "steps"   : steps_history,
        "epsilons": epsilons_history,
    }


# Algoritmo Monte Carlo

In [ ]:
def run_monte_carlo(env, n_episodes: int, gamma: float,
                    eps_start: float, eps_end: float,
                    eps_decay: float, seed: int = 42) -> dict:
    """
    Ejecuta control Monte Carlo on-policy (first-visit) sobre el entorno.

    El algoritmo genera episodios completos con política ε-greedy,
    calcula el retorno G hacia atrás desde el final, y actualiza Q[s][a]
    como la media de todos los retornos first-visit observados para (s,a).

    Parameters
    ----------
    env        : DonkeyKongInverso
    n_episodes : int,   número total de episodios de entrenamiento
    gamma      : float, factor de descuento ∈ (0, 1]
    eps_start  : float, exploración inicial
    eps_end    : float, exploración mínima
    eps_decay  : float, factor multiplicativo de decaimiento por episodio
    seed       : int,   semilla para reproducibilidad

    Returns
    -------
    dict con:
        Q        : np.ndarray (n_states, 4), tabla de valores acción-estado
        rewards  : list[float], recompensa total por episodio
        steps    : list[int],   pasos por episodio
        epsilons : list[float], valor de ε por episodio
    """
    np.random.seed(seed)

    # ── Inicialización ────────────────────────────────────────────────────────
    Q       = init_Q(env.n_states)          # Q[s][a] = 0 para todo s, a
    returns = defaultdict(list)             # (s_idx, a) → lista de retornos G

    rewards_history  = []
    steps_history    = []
    epsilons_history = []

    epsilon = eps_start

    MAX_STEPS = 500   # límite de seguridad por episodio

    for ep in range(n_episodes):

        # ── Generar episodio completo con política ε-greedy ───────────────────
        episode = []          # lista de tuplas (s_idx, action, reward)
        state   = env.reset()
        done    = False
        n_steps = 0

        while not done and n_steps < MAX_STEPS:
            s_idx  = env.state_to_idx(state)
            action = eps_greedy_action(Q, s_idx, epsilon)

            next_state, reward, done = env.step(action)

            episode.append((s_idx, action, reward))

            state   = next_state
            n_steps += 1

        # ── Calcular retornos G y actualizar Q (first-visit) ──────────────────
        G        = 0.0
        visited  = set()    # pares (s_idx, action) ya procesados en este episodio

        for s_idx, action, reward in reversed(episode):
            # Acumular retorno descontado desde el final del episodio
            G = gamma * G + reward

            # First-visit: solo actualizar la primera vez que aparece (s, a)
            if (s_idx, action) not in visited:
                visited.add((s_idx, action))
                returns[(s_idx, action)].append(G)
                # Q[s][a] = media de todos los retornos observados para (s,a)
                Q[s_idx][action] = np.mean(returns[(s_idx, action)])

        # ── Registrar métricas del episodio ───────────────────────────────────
        total_reward = sum(r for _, _, r in episode)
        rewards_history.append(total_reward)
        steps_history.append(n_steps)
        epsilons_history.append(epsilon)

        # Decaer epsilon al final del episodio (no durante)
        epsilon = max(eps_end, epsilon * eps_decay)

    return {
        "Q"       : Q,
        "rewards" : rewards_history,
        "steps"   : steps_history,
        "epsilons": epsilons_history,
    }


print("Función run_monte_carlo definida.")

# Entrenamiento multi-run (5 seeds)

In [ ]:
def train_multirun(train_fn, env, config: dict, label: str = "") -> dict:
    """
    Ejecuta train_fn N veces con distintas semillas y agrega resultados.

    Parameters
    ----------
    train_fn : callable, función de entrenamiento (run_monte_carlo o run_ql)
    env      : DonkeyKongInverso
    config   : dict, CONFIG global
    label    : str,  nombre del algoritmo para el log

    Returns
    dict con:
        Q_best   : tabla Q del run con mayor recompensa media final
        rewards_mean : np.ndarray, media de recompensas por episodio
        rewards_std  : np.ndarray, desviación estándar
        steps_mean   : np.ndarray
        metrics      : dict devuelto por compute_metrics sobre el mejor run
        all_rewards  : list de listas (una por seed)
    """
    all_rewards = []
    all_steps   = []
    all_Q       = []

    for i, seed in enumerate(config["seeds"]):
        print(f"  [{label}] Run {i+1}/{config['n_runs']}  seed={seed} ...", end=" ")

        result = train_fn(
            env,
            n_episodes = config["n_episodes"],
            gamma      = config["gamma"],
            eps_start  = config["eps_start"],
            eps_end    = config["eps_end"],
            eps_decay  = config["eps_decay"],
            seed       = seed,
        )

        all_rewards.append(result["rewards"])
        all_steps.append(result["steps"])
        all_Q.append(result["Q"])

        mean_last = np.mean(result["rewards"][-config["eval_last_n"]:])
        print(f"recompensa media últimos {config['eval_last_n']} eps: {mean_last:.2f}")

    # Agregar entre runs
    rewards_arr  = np.array(all_rewards)   # shape (n_runs, n_episodes)
    steps_arr    = np.array(all_steps)

    rewards_mean = rewards_arr.mean(axis=0)
    rewards_std  = rewards_arr.std(axis=0)
    steps_mean   = steps_arr.mean(axis=0)

    # Mejor run = el que tiene mayor recompensa media en los últimos eval_last_n
    best_run_idx = int(np.argmax(
        [np.mean(r[-config["eval_last_n"]:]) for r in all_rewards]
    ))
    Q_best = all_Q[best_run_idx]

    metrics = compute_metrics(
        all_rewards[best_run_idx],
        all_steps[best_run_idx],
        last_n=config["eval_last_n"],
    )

    return {
        "Q_best"      : Q_best,
        "rewards_mean": rewards_mean,
        "rewards_std" : rewards_std,
        "steps_mean"  : steps_mean,
        "metrics"     : metrics,
        "all_rewards" : all_rewards,
    }


print("Función train_multirun definida.")

# Ejecución MC y QL

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  ENTRENAMIENTO — Monte Carlo (on-policy, first-visit)
# ════════════════════════════════════════════════════════════════════════════

print("═" * 55)
print("  Entrenando Monte Carlo...")
print(f"  {CONFIG['n_episodes']} episodios × {CONFIG['n_runs']} seeds")
print("═" * 55)

mc_results = train_multirun(
    train_fn = run_monte_carlo,
    env      = env,
    config   = CONFIG,
    label    = "MC",
)

Q_mc = mc_results["Q_best"]

print()
print("Entrenamiento MC completado.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  ENTRENAMIENTO — Q_LEARNING
# ════════════════════════════════════════════════════════════════════════════

print("="*55)
print("  Entrenando Q-Learning...")
print(f"  {CONFIG['n_episodes']} episodios × {CONFIG['n_runs']} seeds")
print("="*55)

ql_results = train_multirun(
    train_fn = run_q_learning,
    env      = env,
    config   = CONFIG,
    label    = "QL",
)

Q_ql = ql_results["Q_best"]

print()
print(" Entrenamiento QL completado.")

 # Curvas comparativas (llamando a la funcion de arriba)

In [ ]:
plot_learning_curves(
    rewards_mc = mc_results["rewards_mean"],
    std_mc     = mc_results["rewards_std"],
    rewards_ql = ql_results["rewards_mean"],
    std_ql     = ql_results["rewards_std"],
    title      = "Monte Carlo vs Q‑Learning"
)

# Ampliamos episodios de MC para intentar garantizar convergencia

> El entrenamiento anterior (5000 eps) muestra que MC no converge de forma
> consistente en todas las seeds. A continuación se reentrena con 20000 episodios.
> `mc_results` y `Q_mc` quedan actualizados con este nuevo entrenamiento.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  ENTRENAMIENTO — Monte Carlo (on-policy, first-visit)
#  Nota: Como MC requiere más episodios que QL para converger de forma consistente,
#  consecuencia directa de aprender solo al final de episodios completos.
# ════════════════════════════════════════════════════════════════════════════

CONFIG_MC = CONFIG.copy()
CONFIG_MC["n_episodes"] = 20000   # mayor que QL (5000) para intentar garantizar convergencia

print("═" * 55)
print("  Entrenando Monte Carlo...")
print(f"  {CONFIG_MC['n_episodes']} episodios × {CONFIG_MC['n_runs']} seeds")
print("═" * 55)

mc_results = train_multirun(
    train_fn = run_monte_carlo,
    env      = env,
    config   = CONFIG_MC,
    label    = "MC",
)

Q_mc = mc_results["Q_best"]

print()
print("Entrenamiento MC completado.")

# Seguimiento de la política Greedy en Monte Carlo

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  RUTA GREEDY — seguimiento de la política aprendida
# ════════════════════════════════════════════════════════════════════════════

def extract_greedy_path(env, Q: np.ndarray, max_steps: int = 100) -> list:
    """
    Sigue la política greedy (ε=0) desde env.start hasta env.goal.

    No usa exploración: en cada paso toma la acción con mayor Q[s].
    Si el agente entra en bucle o supera max_steps, devuelve la ruta parcial.

    Returns
    -------
    list de tuplas (fila, col) que forman la ruta, incluyendo inicio y fin.
    """
    state = env.reset()
    path  = [state]
    done  = False
    steps = 0

    while not done and steps < max_steps:
        s_idx  = env.state_to_idx(state)
        action = int(np.argmax(Q[s_idx]))        # acción greedy pura

        next_state, _, done = env.step(action)
        path.append(next_state)
        state = next_state
        steps += 1

        # Detectar bucle: el mismo estado aparece más de 2 veces
        if path.count(state) > 2:
            print("    Bucle detectado en la ruta greedy.")
            break

    if state == env.goal:
        print(f"   Ruta greedy encontrada en {len(path)-1} pasos.")
    else:
        print("   La política greedy no alcanzó la meta.")

    return path


# ── Ruta greedy de Monte Carlo ────────────────────────────────────────────────
path_mc = extract_greedy_path(env, Q_mc)
print(f"  Ruta MC: {' → '.join(str(s) for s in path_mc)}")

# ── Mapa con política y ruta ──────────────────────────────────────────────────
plot_map(
    env,
    Q     = Q_mc,
    path  = path_mc,
    title = " Monte Carlo — Política greedy aprendida",
)

# Experimento ε fijo vs decaimiento

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  EXPERIMENTO — Impacto del decaimiento de ε (Pregunta 3 de análisis)
# ════════════════════════════════════════════════════════════════════════════

CONFIG_MC_EPS = CONFIG.copy()
CONFIG_MC_EPS["n_episodes"] = 20000

# ε fijo en 0.20 durante todos los episodios
result_eps_fixed = run_monte_carlo(
    env,
    n_episodes = CONFIG_MC_EPS["n_episodes"],
    gamma      = CONFIG["gamma"],
    eps_start  = 0.20,
    eps_end    = 0.20,    # mínimo igual al inicio → nunca decae
    eps_decay  = 1.00,    # factor 1.0 → sin cambio
    seed       = 99,      # seed en la que convergen ambos algoritmos
)

# ε con decaimiento estándar (mismo seed para comparación justa)
result_eps_decay = run_monte_carlo(
    env,
    n_episodes = CONFIG_MC_EPS["n_episodes"],
    gamma      = CONFIG["gamma"],
    eps_start  = CONFIG["eps_start"],
    eps_end    = CONFIG["eps_end"],
    eps_decay  = CONFIG["eps_decay"],
    seed       = 99,
)

# ── Gráfica comparativa ───────────────────────────────────────────────────────
w   = CONFIG["smoothing_window"]
fig, ax = plt.subplots(figsize=(11, 4))

for rewards, color, label in [
    (result_eps_fixed["rewards"],  DS["col_hole"], f"ε fijo = 0.20"),
    (result_eps_decay["rewards"],  DS["col_mc"],   f"ε con decaimiento → {CONFIG['eps_end']}"),
]:
    smoothed = smooth(rewards, w)
    x        = np.arange(w - 1, len(rewards))
    ax.plot(rewards,  color=color, alpha=0.10, linewidth=1.0)
    ax.plot(x, smoothed, color=color, linewidth=2.2, label=label)

ax.axhline(0, color=DS["col_subtext"], linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("Episodio",         fontfamily=DS["font_mono"])
ax.set_ylabel("Recompensa total", fontfamily=DS["font_mono"])
ax.set_title("MC — Impacto del decaimiento de ε (Pregunta 3)",
             fontfamily=DS["font_mono"], fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, axis="y", alpha=0.3)

eval_n   = CONFIG["eval_last_n"]
m_fixed  = np.mean(result_eps_fixed["rewards"][-eval_n:])
m_decay  = np.mean(result_eps_decay["rewards"][-eval_n:])

ax.text(0.01, 0.05,
        f"Media últimos {eval_n} eps → ε fijo: {m_fixed:.1f}  |  ε decay: {m_decay:.1f}",
        transform=ax.transAxes, fontsize=8.5,
        color=DS["col_subtext"], fontfamily=DS["font_mono"])

plt.tight_layout()
plt.show()

print(f"  ε fijo  — recompensa media últimos {eval_n} eps: {m_fixed:.2f}")
print(f"  ε decay — recompensa media últimos {eval_n} eps: {m_decay:.2f}")

# Entorno estocástico en ambos algoritmos

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  EXPERIMENTO — Entorno estocástico, slip=10% (Pregunta 4 de análisis)
# ════════════════════════════════════════════════════════════════════════════

print("Entrenando MC sobre entorno estocástico...")
mc_stoch_results = train_multirun(
    train_fn = run_monte_carlo,
    env      = env_stochastic,   # ← entorno con ruido
    config   = CONFIG_MC,        # ← 20000 eps para MC
    label    = "MC-stoch",
)

print("\nEntrenando QL sobre entorno estocástico...")
ql_stoch_results = train_multirun(
    train_fn = run_q_learning,
    env      = env_stochastic,   # ← entorno con ruido
    config   = CONFIG,           # ← 5000 eps para QL
    label    = "QL-stoch",
)

# ── Curvas comparativas ───────────────────────────────────────────────────────
plot_learning_curves(
    rewards_mc = mc_stoch_results["rewards_mean"],
    rewards_ql = ql_stoch_results["rewards_mean"],
    std_mc     = mc_stoch_results["rewards_std"],
    std_ql     = ql_stoch_results["rewards_std"],
    title      = "MC vs QL — Entorno estocástico (slip=10%) — Pregunta 4",
)

# ── Tabla comparativa ─────────────────────────────────────────────────────────
print_metrics_table({
    "MC": mc_stoch_results["metrics"],
    "QL": ql_stoch_results["metrics"],
})

# Comparativa completa de Monte Carlo y Q_Learning

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Comparativa completa MC vs Q-Learning
# ════════════════════════════════════════════════════════════════════════════

# ── Ruta greedy QL ────────────────────────────────────────────────────────────
path_ql = extract_greedy_path(env, Q_ql)
print(f"  Ruta QL: {' → '.join(str(s) for s in path_ql)}")

# ── Tabla de métricas comparativa ────────────────────────────────────────────
print_metrics_table({
    "MC": mc_results["metrics"],
    "QL": ql_results["metrics"],
})

# ── Mapas de política lado a lado ────────────────────────────────────────────
plot_map(
    env,
    Q     = Q_mc,
    path  = path_mc,
    title = " Monte Carlo — Política greedy",
)

plot_map(
    env,
    Q     = Q_ql,
    path  = path_ql,
    title = " Q-Learning — Política greedy",
)

---

## Preguntas de análisis

---

### 1. Convergencia: ¿Qué algoritmo alcanza antes una política consistente?

Q-Learning converge más rápido que Monte Carlo en este entorno. La razón es
estructural: Q-Learning actualiza la tabla Q en **cada paso** usando bootstrapping
(aprovecha estimaciones ya existentes), mientras que Monte Carlo necesita esperar
al **final del episodio completo** para propagar las recompensas hacia atrás. En un
entorno donde alcanzar la meta por primera vez puede costar cientos de pasos
aleatorios, MC tarda más en recibir señal útil. Q-Learning, al actualizar
paso a paso, ajusta su estimación de Q incluso en episodios fallidos, acelerando
el aprendizaje inicial.

---

### 2. Calidad de la solución: ¿Ambos encuentran la misma ruta óptima?

En este entorno ambos algoritmos convergen a la misma ruta óptima de 9 pasos:
`(0,0)→(0,1)→(0,2)→(3,0)→(3,1)→(3,2)→(3,3)→(5,3)→(5,4)→(5,5)`.
El agente avanza horizontalmente hasta la escalera en `(0,2)`, desciende a `(3,0)`,
recorre la fila 3 hasta la escalera en `(3,3)`, baja a `(5,3)` y avanza hasta la meta.
Que ambos encuentren la misma ruta confirma que han convergido a la política óptima
global: Q-Learning por su actualización off-policy con el máximo del estado siguiente,
y Monte Carlo por haber acumulado suficientes retornos positivos en las seeds que sí
convergieron.

---

### 3. Exploración: ¿Qué ocurre si ε no decae?

Con ε fijo en 0.20, el agente sigue explorando aleatoriamente el 20% de los pasos
incluso al final del entrenamiento, cuando ya ha aprendido una buena política. Esto
introduce ruido permanente: episodios que de otro modo llegarían a la meta acaban
en agujero o timeout por una acción aleatoria desafortunada. El resultado visible
en la gráfica es que la recompensa media de los últimos 500 episodios es
significativamente menor con ε fijo que con decaimiento. El agente aprende
correctamente, pero no puede *explotar* su conocimiento de forma consistente.

---

### 4. Estocasticidad simulada: ¿Cuál se comporta mejor con slip=10%?

Q-Learning se adapta mejor al entorno estocástico. Al actualizar en cada paso
con la ecuación TD, incorpora rápidamente la incertidumbre del entorno en sus
estimaciones de Q. Monte Carlo, en cambio, promedia retornos de episodios
completos: un único resbalón en cualquier punto del episodio contamina el retorno
G de todos los pares (s,a) anteriores, introduciendo mayor varianza en las
actualizaciones. En entornos estocásticos, la actualización paso a paso de
Q-Learning es más robusta porque aísla mejor el efecto del ruido puntual.

---

### 5. Análisis de la política: ¿El agente evita agujeros y usa escaleras?

La política greedy aprendida por Q-Learning evita correctamente los agujeros en
`(2,1)` y `(2,3)`: el agente aprende a no tomar las acciones que conducen a esas
celdas, rodeándolas por la fila 0 y la fila 3. Respecto a las escaleras, el agente
las utiliza activamente: en `(0,2)` toma la escalera hacia `(3,0)`, y en `(3,3)`
toma la escalera hacia `(5,3)`, reduciendo el recorrido a solo 9 pasos. Ambas
políticas (MC y QL) son idénticas, lo que confirma que la tabla Q ha capturado
correctamente la estructura del entorno en los runs que convergieron.

---

## Posibles mejoras

### 1. Entorno más rico
El mapa actual es determinista y pequeño (6×6). Una mejora natural sería
aumentar el tamaño del grid, añadir más agujeros y escaleras, o introducir
recompensas intermedias (por ejemplo, +1 al bajar de nivel correctamente).
Esto haría el problema más desafiante y permitiría observar diferencias más
marcadas entre MC y QL.

### 2. SARSA como tercer algoritmo
El enunciado menciona SARSA como alternativa on-policy con bootstrapping.
Añadirlo a la comparativa sería directo (solo cambia una línea en la
actualización) y permitiría distinguir empíricamente entre on-policy con
episodios completos (MC), on-policy con TD (SARSA) y off-policy con TD (QL).

### 3. Decaimiento adaptativo de ε
El decaimiento multiplicativo fijo (`ε × 0.999`) no tiene en cuenta el
progreso real del agente. Una mejora sería reducir ε más agresivamente
cuando la tasa de éxito reciente supera un umbral, y más lentamente cuando
el agente sigue fallando. Esto resolvería el problema de convergencia de MC
en las seeds problemáticas sin necesitar 20000 episodios.

### 4. Inicialización optimista de Q
Inicializar Q con valores positivos altos (por ejemplo, +10) en lugar de
ceros fuerza al agente a explorar todos los estados al menos una vez antes
de explotar. En MC esto podría acelerar la convergencia en las seeds
problemáticas, ya que el agente tendría incentivo a visitar estados no
explorados en lugar de repetir los ya conocidos.

### 5. Visualización interactiva
El notebook podría incluir una animación cuadro a cuadro del agente
siguiendo la política greedy aprendida, usando `matplotlib.animation`.
Esto haría la presentación más visual e intuitiva para mostrar en clase.